# Bayesian Optimization (GP-based) baseline

This notebook runs the **Bayesian Optimization (BO)** baseline using
Gaussian Processes with the Expected Improvement acquisition function,
as implemented in Fernando Nogueira's `bayesian-optimization` package
(https://bayesian-optimization.github.io/BayesianOptimization/).

The protocol mirrors the evolutionary baselines (GA, SADE, PSO, GWO):

- Same 7-dimensional categorical search space (encoded as integer indices)
- Same fitness function: `1 - val_accuracy` after `eval_epochs` epochs (internally BO maximises `val_accuracy` because the library is a maximizer; the EA-style fitness is logged alongside for consistency)
- Same budget: `init_points + n_iter = 20 + 200 = 220` evaluations (matches the EA's 20 initial individuals + 10 generations of 20 offspring)
- N = 5 independent runs with seeds `0..4`
- Results saved under `code/colab/results/bo/<dataset>/` in the same JSON/CSV format as the EA runs, so the aggregation scripts work unchanged.

Install the dependency once before running:
```
pip install bayesian-optimization>=2.0.0
```

## 1. Imports and setup

In [1]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / 'utils').exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / 'utils').exists():
    sys.path.insert(0, str(ROOT.parent))

from utils.datasets      import getDataset
from utils.optimizer_bo  import runBOOptimization

I0000 00:00:1779317462.558275      55 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779317464.660216      55 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779317469.340318      55 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 2. Configuration

In [2]:
SEEDS       = [0, 1, 2, 3, 4]   # 5 independent runs
EVAL_EPOCHS = 1                  # same as the EA baselines
N_ITER      = 200                # BO-guided iterations
INIT_POINTS = 20                 # random initialization (== EA pop_size)
XI          = 0.01               # Expected Improvement exploration trade-off

RESULTS_ROOT = ROOT / 'colab' / 'results'
if not RESULTS_ROOT.exists():
    RESULTS_ROOT = ROOT.parent / 'code' / 'colab' / 'results'

print('SEEDS       :', SEEDS)
print('Total evals :', INIT_POINTS + N_ITER, f'(= {INIT_POINTS} init + {N_ITER} BO)')
print('RESULTS_ROOT:', RESULTS_ROOT)

SEEDS       : [0, 1, 2, 3, 4]
Total evals : 220 (= 20 init + 200 BO)
RESULTS_ROOT: /workspace/code/colab/results


## 3. Run BO on MNIST (5 seeds)

In [3]:
data_mnist = getDataset('mnist')
print('train:', data_mnist['x_train'].shape, '| test:', data_mnist['x_test'].shape)

  MNIST loaded:
    Train      :  60000 samples
    Test       :  10000 samples
    Shape      : (28, 28, 1)
    Classes    : ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
train: (60000, 28, 28, 1) | test: (10000, 28, 28, 1)


In [4]:
results_mnist = []
out_dir = RESULTS_ROOT / 'bo' / 'mnist'
out_dir.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    print(f'\n##### BO seed = {seed}  (MNIST) #####\n')
    best, run = runBOOptimization(
        data_mnist,
        eval_epochs = EVAL_EPOCHS,
        n_iter      = N_ITER,
        init_points = INIT_POINTS,
        xi          = XI,
        seed        = seed,
        results_dir = str(out_dir),
    )
    results_mnist.append({'seed': seed, 'best_val_acc': run['best']['val_accuracy']})

df = pd.DataFrame(results_mnist)
print('\nMNIST BO summary across', len(SEEDS), 'seeds:')
print(df.to_string(index=False))
print(f"\nmean +/- std: {df['best_val_acc'].mean():.4f} +/- {df['best_val_acc'].std(ddof=1):.4f}")


##### BO seed = 0  (MNIST) #####


  OPTIMIZATION: BO  (bayesian-optimization, GP + Expected Improvement)
  Trials: 220  (random init: 20  +  BO-guided: 200)
  Epochs per evaluation: 1
  Search space dimensions: 7 (batch, base_filters, n_blocks, dense, opt, pool, kernel)
  Fixed: lr=0.001  dropout=0.5  activation=relu  batch_norm=True
  Seed:  0



/workspace/code/utils/optimizer_bo.py:288: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  bo = BayesianOptimization(


  [   1] val_acc=0.9746  best=0.9746  bs=32  f0=64  blocks=2  dense=256  opt=sgd  pool=max  k=3  dt=39s  t=39s  ram=2.24GB
  [   2] val_acc=0.9850  best=0.9850  bs=128  f0=32  blocks=2  dense=128  opt=rmsprop  pool=average  k=3  dt=11s  t=50s  ram=2.32GB
  [   3] val_acc=0.6690  best=0.9850  bs=128  f0=32  blocks=3  dense=256  opt=sgd  pool=average  k=3  dt=13s  t=62s  ram=2.33GB
  [   4] val_acc=0.9685  best=0.9850  bs=64  f0=32  blocks=3  dense=128  opt=sgd  pool=max  k=5  dt=24s  t=86s  ram=2.35GB
  [   5] val_acc=0.9856  best=0.9856  bs=32  f0=128  blocks=3  dense=128  opt=sgd  pool=average  k=5  dt=62s  t=148s  ram=2.35GB
  [   6] val_acc=0.9715  best=0.9856  bs=128  f0=32  blocks=3  dense=512  opt=rmsprop  pool=average  k=3  dt=12s  t=161s  ram=2.35GB
  [   7] val_acc=0.8532  best=0.9856  bs=128  f0=32  blocks=2  dense=128  opt=sgd  pool=average  k=3  dt=10s  t=170s  ram=2.35GB
  [   8] val_acc=0.9715  best=0.9856  bs=32  f0=32  blocks=3  dense=128  opt=sgd  pool=max  k=3  dt=46s

/workspace/code/utils/optimizer_bo.py:288: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  bo = BayesianOptimization(


  [   1] val_acc=0.9437  best=0.9437  bs=64  f0=32  blocks=2  dense=256  opt=sgd  pool=average  k=3  dt=18s  t=18s  ram=2.45GB
  [   2] val_acc=0.9716  best=0.9716  bs=32  f0=64  blocks=2  dense=256  opt=adamw  pool=max  k=5  dt=38s  t=56s  ram=2.45GB
  [   3] val_acc=0.9882  best=0.9882  bs=128  f0=32  blocks=2  dense=256  opt=rmsprop  pool=max  k=5  dt=11s  t=67s  ram=2.46GB
  [   4] val_acc=0.9874  best=0.9882  bs=32  f0=128  blocks=2  dense=256  opt=rmsprop  pool=max  k=3  dt=39s  t=106s  ram=2.46GB
  [   5] val_acc=0.9709  best=0.9882  bs=64  f0=64  blocks=3  dense=512  opt=adamw  pool=max  k=5  dt=28s  t=134s  ram=2.46GB
  [   6] val_acc=0.9257  best=0.9882  bs=64  f0=64  blocks=3  dense=256  opt=rmsprop  pool=average  k=5  dt=26s  t=160s  ram=2.46GB
  [   7] val_acc=0.9796  best=0.9882  bs=32  f0=32  blocks=3  dense=128  opt=adamw  pool=average  k=5  dt=51s  t=211s  ram=2.46GB
  [   8] val_acc=0.9824  best=0.9882  bs=128  f0=64  blocks=2  dense=512  opt=rmsprop  pool=average  k=

...

  [   1] val_acc=0.9819  best=0.9819  bs=32  f0=64  blocks=2  dense=512  opt=rmsprop  pool=average  k=3  dt=37s  t=37s  ram=2.45GB
  [   2] val_acc=0.9897  best=0.9897  bs=128  f0=64  blocks=3  dense=256  opt=rmsprop  pool=max  k=3  dt=16s  t=52s  ram=2.47GB
  [   3] val_acc=0.9890  best=0.9897  bs=32  f0=64  blocks=3  dense=512  opt=adamw  pool=max  k=5  dt=50s  t=103s  ram=2.47GB
  [   4] val_acc=0.9519  best=0.9897  bs=128  f0=128  blocks=3  dense=512  opt=adamw  pool=average  k=5  dt=37s  t=140s  ram=2.47GB
  [   5] val_acc=0.9859  best=0.9897  bs=64  f0=32  blocks=2  dense=512  opt=rmsprop  pool=average  k=5  dt=19s  t=159s  ram=2.47GB
  [   6] val_acc=0.9079  best=0.9897  bs=64  f0=128  blocks=2  dense=128  opt=rmsprop  pool=max  k=3  dt=24s  t=183s  ram=2.47GB
  [   7] val_acc=0.9872  best=0.9897  bs=32  f0=128  blocks=3  dense=256  opt=rmsprop  pool=max  k=3  dt=53s  t=237s  ram=2.47GB
  [   8] val_acc=0.9834  best=0.9897  bs=64  f0=128  blocks=3  dense=256  opt=sgd  pool=max  

/workspace/code/utils/optimizer_bo.py:288: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  bo = BayesianOptimization(


  [   1] val_acc=0.9766  best=0.9766  bs=128  f0=32  blocks=3  dense=128  opt=adamw  pool=max  k=5  dt=14s  t=14s  ram=2.46GB
  [   2] val_acc=0.9793  best=0.9793  bs=64  f0=128  blocks=3  dense=256  opt=sgd  pool=max  k=5  dt=39s  t=53s  ram=2.47GB
  [   3] val_acc=0.6807  best=0.9793  bs=128  f0=32  blocks=2  dense=128  opt=sgd  pool=average  k=3  dt=10s  t=63s  ram=2.47GB
  [   4] val_acc=0.9895  best=0.9895  bs=32  f0=128  blocks=3  dense=512  opt=rmsprop  pool=max  k=3  dt=53s  t=116s  ram=2.47GB
  [   5] val_acc=0.9785  best=0.9895  bs=32  f0=64  blocks=3  dense=512  opt=sgd  pool=max  k=3  dt=48s  t=164s  ram=2.47GB
  [   6] val_acc=0.9322  best=0.9895  bs=64  f0=64  blocks=2  dense=128  opt=rmsprop  pool=average  k=3  dt=19s  t=183s  ram=2.47GB
  [   7] val_acc=0.9792  best=0.9895  bs=64  f0=32  blocks=3  dense=128  opt=rmsprop  pool=average  k=5  dt=25s  t=208s  ram=2.47GB
  [   8] val_acc=0.9497  best=0.9895  bs=128  f0=32  blocks=2  dense=512  opt=adamw  pool=max  k=5  dt=11

/workspace/code/utils/optimizer_bo.py:288: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  bo = BayesianOptimization(


  [   1] val_acc=0.2790  best=0.2790  bs=128  f0=128  blocks=3  dense=256  opt=sgd  pool=max  k=5  dt=35s  t=35s  ram=2.48GB
  [   2] val_acc=0.9667  best=0.9667  bs=32  f0=128  blocks=3  dense=512  opt=adamw  pool=average  k=5  dt=67s  t=102s  ram=2.48GB
  [   3] val_acc=0.9655  best=0.9667  bs=128  f0=64  blocks=2  dense=128  opt=rmsprop  pool=max  k=3  dt=11s  t=114s  ram=2.48GB
  [   4] val_acc=0.9197  best=0.9667  bs=128  f0=128  blocks=2  dense=128  opt=sgd  pool=average  k=5  dt=25s  t=139s  ram=2.48GB
  [   5] val_acc=0.9813  best=0.9813  bs=128  f0=64  blocks=2  dense=128  opt=rmsprop  pool=max  k=5  dt=13s  t=153s  ram=2.48GB
  [   6] val_acc=0.9772  best=0.9813  bs=128  f0=32  blocks=3  dense=512  opt=rmsprop  pool=average  k=3  dt=13s  t=165s  ram=2.48GB
  [   7] val_acc=0.9678  best=0.9813  bs=32  f0=128  blocks=3  dense=256  opt=adamw  pool=max  k=5  dt=68s  t=233s  ram=2.48GB
  [   8] val_acc=0.9821  best=0.9821  bs=32  f0=128  blocks=3  dense=512  opt=sgd  pool=average 

## 4. Run BO on CIFAR-10 (5 seeds)

In [5]:
data_cifar = getDataset('cifar10')
print('train:', data_cifar['x_train'].shape, '| test:', data_cifar['x_test'].shape)

  CIFAR-10 loaded:
    Train      :  50000 samples
    Test       :  10000 samples
    Shape      : (32, 32, 3)
    Classes    : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
train: (50000, 32, 32, 3) | test: (10000, 32, 32, 3)


In [6]:
results_cifar = []
out_dir = RESULTS_ROOT / 'bo' / 'cifar10'
out_dir.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    print(f'\n##### BO seed = {seed}  (CIFAR-10) #####\n')
    best, run = runBOOptimization(
        data_cifar,
        eval_epochs = EVAL_EPOCHS,
        n_iter      = N_ITER,
        init_points = INIT_POINTS,
        xi          = XI,
        seed        = seed,
        results_dir = str(out_dir),
    )
    results_cifar.append({'seed': seed, 'best_val_acc': run['best']['val_accuracy']})

df = pd.DataFrame(results_cifar)
print('\nCIFAR-10 BO summary across', len(SEEDS), 'seeds:')
print(df.to_string(index=False))
print(f"\nmean +/- std: {df['best_val_acc'].mean():.4f} +/- {df['best_val_acc'].std(ddof=1):.4f}")


##### BO seed = 0  (CIFAR-10) #####


  OPTIMIZATION: BO  (bayesian-optimization, GP + Expected Improvement)
  Trials: 220  (random init: 20  +  BO-guided: 200)
  Epochs per evaluation: 1
  Search space dimensions: 7 (batch, base_filters, n_blocks, dense, opt, pool, kernel)
  Fixed: lr=0.001  dropout=0.5  activation=relu  batch_norm=True
  Seed:  0



/workspace/code/utils/optimizer_bo.py:288: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  bo = BayesianOptimization(


  [   1] val_acc=0.4647  best=0.4647  bs=32  f0=64  blocks=2  dense=256  opt=sgd  pool=max  k=3  dt=32s  t=32s  ram=3.15GB
  [   2] val_acc=0.2508  best=0.4647  bs=128  f0=32  blocks=2  dense=128  opt=rmsprop  pool=average  k=3  dt=9s  t=41s  ram=3.16GB
  [   3] val_acc=0.1870  best=0.4647  bs=128  f0=32  blocks=3  dense=256  opt=sgd  pool=average  k=3  dt=13s  t=54s  ram=3.16GB
  [   4] val_acc=0.4254  best=0.4647  bs=64  f0=32  blocks=3  dense=128  opt=sgd  pool=max  k=5  dt=22s  t=75s  ram=3.17GB
  [   5] val_acc=0.5195  best=0.5195  bs=32  f0=128  blocks=3  dense=128  opt=sgd  pool=average  k=5  dt=55s  t=130s  ram=3.17GB
  [   6] val_acc=0.4198  best=0.5195  bs=128  f0=32  blocks=3  dense=512  opt=rmsprop  pool=average  k=3  dt=11s  t=141s  ram=3.17GB
  [   7] val_acc=0.2339  best=0.5195  bs=128  f0=32  blocks=2  dense=128  opt=sgd  pool=average  k=3  dt=10s  t=151s  ram=3.17GB
  [   8] val_acc=0.4362  best=0.5195  bs=32  f0=32  blocks=3  dense=128  opt=sgd  pool=max  k=3  dt=43s 

/workspace/code/utils/optimizer_bo.py:288: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  bo = BayesianOptimization(


  [   1] val_acc=0.3801  best=0.3801  bs=64  f0=32  blocks=2  dense=256  opt=sgd  pool=average  k=3  dt=16s  t=16s  ram=3.21GB
  [   2] val_acc=0.3062  best=0.3801  bs=32  f0=64  blocks=2  dense=256  opt=adamw  pool=max  k=5  dt=35s  t=50s  ram=3.21GB
  [   3] val_acc=0.3808  best=0.3808  bs=128  f0=32  blocks=2  dense=256  opt=rmsprop  pool=max  k=5  dt=10s  t=60s  ram=3.22GB
  [   4] val_acc=0.2697  best=0.3808  bs=32  f0=128  blocks=2  dense=256  opt=rmsprop  pool=max  k=3  dt=37s  t=97s  ram=3.22GB
  [   5] val_acc=0.2782  best=0.3808  bs=64  f0=64  blocks=3  dense=512  opt=adamw  pool=max  k=5  dt=27s  t=124s  ram=3.22GB
  [   6] val_acc=0.2953  best=0.3808  bs=64  f0=64  blocks=3  dense=256  opt=rmsprop  pool=average  k=5  dt=25s  t=148s  ram=3.22GB
  [   7] val_acc=0.1473  best=0.3808  bs=32  f0=32  blocks=3  dense=128  opt=adamw  pool=average  k=5  dt=43s  t=192s  ram=3.22GB
  [   8] val_acc=0.2577  best=0.3808  bs=128  f0=64  blocks=2  dense=512  opt=rmsprop  pool=average  k=5

...

  [   1] val_acc=0.4339  best=0.4339  bs=32  f0=64  blocks=2  dense=512  opt=rmsprop  pool=average  k=3  dt=31s  t=31s  ram=3.21GB
  [   2] val_acc=0.2309  best=0.4339  bs=128  f0=64  blocks=3  dense=256  opt=rmsprop  pool=max  k=3  dt=14s  t=45s  ram=3.22GB
  [   3] val_acc=0.3053  best=0.4339  bs=32  f0=64  blocks=3  dense=512  opt=adamw  pool=max  k=5  dt=43s  t=88s  ram=3.22GB
  [   4] val_acc=0.3022  best=0.4339  bs=128  f0=128  blocks=3  dense=512  opt=adamw  pool=average  k=5  dt=39s  t=127s  ram=3.22GB
  [   5] val_acc=0.3091  best=0.4339  bs=64  f0=32  blocks=2  dense=512  opt=rmsprop  pool=average  k=5  dt=16s  t=143s  ram=3.22GB
  [   6] val_acc=0.1000  best=0.4339  bs=64  f0=128  blocks=2  dense=128  opt=rmsprop  pool=max  k=3  dt=22s  t=165s  ram=3.22GB
  [   7] val_acc=0.1000  best=0.4339  bs=32  f0=128  blocks=3  dense=256  opt=rmsprop  pool=max  k=3  dt=45s  t=210s  ram=3.22GB
  [   8] val_acc=0.5013  best=0.5013  bs=64  f0=128  blocks=3  dense=256  opt=sgd  pool=max  k

/workspace/code/utils/optimizer_bo.py:288: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  bo = BayesianOptimization(


  [   1] val_acc=0.3832  best=0.3832  bs=128  f0=32  blocks=3  dense=128  opt=adamw  pool=max  k=5  dt=13s  t=13s  ram=3.22GB
  [   2] val_acc=0.4908  best=0.4908  bs=64  f0=128  blocks=3  dense=256  opt=sgd  pool=max  k=5  dt=41s  t=54s  ram=3.22GB
  [   3] val_acc=0.2073  best=0.4908  bs=128  f0=32  blocks=2  dense=128  opt=sgd  pool=average  k=3  dt=11s  t=64s  ram=3.22GB
  [   4] val_acc=0.3653  best=0.4908  bs=32  f0=128  blocks=3  dense=512  opt=rmsprop  pool=max  k=3  dt=51s  t=116s  ram=3.22GB
  [   5] val_acc=0.4992  best=0.4992  bs=32  f0=64  blocks=3  dense=512  opt=sgd  pool=max  k=3  dt=41s  t=157s  ram=3.22GB
  [   6] val_acc=0.3156  best=0.4992  bs=64  f0=64  blocks=2  dense=128  opt=rmsprop  pool=average  k=3  dt=17s  t=175s  ram=3.22GB
  [   7] val_acc=0.3793  best=0.4992  bs=64  f0=32  blocks=3  dense=128  opt=rmsprop  pool=average  k=5  dt=22s  t=196s  ram=3.22GB
  [   8] val_acc=0.2779  best=0.4992  bs=128  f0=32  blocks=2  dense=512  opt=adamw  pool=max  k=5  dt=11

/workspace/code/utils/optimizer_bo.py:288: UserWarning: Non-float parameters are experimental and may not work as expected. Exercise caution when using them and please report any issues you encounter.
  bo = BayesianOptimization(


  [   1] val_acc=0.1436  best=0.1436  bs=128  f0=128  blocks=3  dense=256  opt=sgd  pool=max  k=5  dt=38s  t=38s  ram=3.22GB
  [   2] val_acc=0.3417  best=0.3417  bs=32  f0=128  blocks=3  dense=512  opt=adamw  pool=average  k=5  dt=64s  t=102s  ram=3.22GB
  [   3] val_acc=0.2402  best=0.3417  bs=128  f0=64  blocks=2  dense=128  opt=rmsprop  pool=max  k=3  dt=12s  t=114s  ram=3.25GB
  [   4] val_acc=0.1818  best=0.3417  bs=128  f0=128  blocks=2  dense=128  opt=sgd  pool=average  k=5  dt=27s  t=141s  ram=3.25GB
  [   5] val_acc=0.2332  best=0.3417  bs=128  f0=64  blocks=2  dense=128  opt=rmsprop  pool=max  k=5  dt=13s  t=154s  ram=3.25GB
  [   6] val_acc=0.3711  best=0.3711  bs=128  f0=32  blocks=3  dense=512  opt=rmsprop  pool=average  k=3  dt=12s  t=166s  ram=3.25GB
  [   7] val_acc=0.1434  best=0.3711  bs=32  f0=128  blocks=3  dense=256  opt=adamw  pool=max  k=5  dt=63s  t=229s  ram=3.25GB
  [   8] val_acc=0.5311  best=0.5311  bs=32  f0=128  blocks=3  dense=512  opt=sgd  pool=average 

## 5. Next steps

After both loops finish, the per-run JSON / CSV files have the same schema as the EA baselines. To produce the same aggregated figures and tables (mean +/- std convergence per generation, boxplot of final fitness, summary table), point the existing aggregation scripts at `colab/results/bo/mnist` and `colab/results/bo/cifar10`.